In [ ]:
from mg_sampling import PROJECT_DIR
import seaborn as sns
import pandas as pd
import numpy as np
from ast import literal_eval
import itertools

In [ ]:
systems = ['7ez0_noion', '7ez0_noion_2']
forcefields = ['nMg_tip3p']
methods = ['plain_md', 'replex']
repids = [0, 1, 2, 3]

df = []
for method, system, forcefield, repid in itertools.product(methods, systems, forcefields, repids):
    path = PROJECT_DIR / method / system / forcefield / f'rep{repid:02d}'
    try:
        _df = pd.read_csv(path / 'ion_analysis.csv', na_filter=False, converters={'inner_names': literal_eval, 'inner_resids': literal_eval, 'inner_resnames': literal_eval})
    except FileNotFoundError:
        continue
    _df['method'] = method
    _df['system'] = system.removesuffix('_2')
    _df['forcefield'] = forcefield
    _df['repid'] = repid
    if method == 'plain_md':
        _df['replicate'] = repid
    else:
        _df['replicate'] = 1 if system.endswith('_2') else 0
    df.append(_df)
df = pd.concat(df)
for col in ['name', 'mgrna', 'forcefield', 'system']:
    df[col] = df[col].astype('category')

In [ ]:
grouped = df.groupby(['method', 'system', 'forcefield', 'replicate', 'repid'], observed=True)['time']
n_frames = (grouped.max() / grouped.unique().apply(np.diff).apply(np.min)).astype(int)

In [ ]:
n_frames

In [ ]:
df.query(f"time >= {tstart}").groupby(['name', 'method', 'system', 'forcefield', 'replicate', 'repid'], observed=True).size() / n_frames


In [ ]:
tstart = 0

grouped = df.query(f"time >= {tstart}").groupby(['name', 'n_inner', 'method', 'system', 'forcefield', 'replicate', 'repid'], observed=True).size() / n_frames
grouped = grouped.reset_index(name='occupancy')

g = sns.FacetGrid(grouped, row='name', col='system')
sns.set_style("ticks", {'axes.grid': True})


g.map_dataframe(
    sns.barplot,
    x='n_inner',
    y='occupancy',
    hue='method',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"$n_\text{RNA}$ (inner-sphere)")
g.set_titles("{row_name} ({col_name})")

In [ ]:
grouped = df.query("n_inner > 0 & time % 1000.0 == 0.0").groupby(['time', 'name', 'method', 'system', 'forcefield', 'replicate'], observed=True).size().reset_index(name='counts')
grouped.set_index('time')

g = sns.FacetGrid(grouped, row='name', aspect=4)
sns.set_style("ticks", {'axes.grid': True})

g.map_dataframe(
    sns.lineplot,
    x='time',
    y='counts',
    hue='method',
)

g.add_legend(g._legend_data)
g.set_ylabels("Avg. number")
g.set_xlabels(r"$n_\text{RNA}$ (inner-sphere)")
g.set_titles("{row_name} ({col_name})")